In [1]:
%run ../scripts/notebook_settings_lean.py
from scipy import stats
from horizonplot import horizonplot
from chromwindow import window
import zarr
import allel
pd.options.display.float_format = '{:10,.3g}'.format 

/home/eriks/miniconda3/envs/baboondiversity/lib/python3.8/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.3 when it was built against 1.14.2, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


A streamlined version of rfmix07 in the original notebooks.

In [2]:
def create_paint_df_ind_compressed(df, ind, mapping, ns_map):
    d = {}
    max_pos = int(df.epos.iloc[-1])
    df = df.loc[df[ind].shift() != df[ind]].reset_index()
    df["epos"] = pd.concat([df.spos.iloc[1:], pd.Series(max_pos)], ignore_index=True)
    d["pos"] = df.spos
    d["end_pos"] = df.epos -1
    d["length"] = df.epos - df.spos
    d["reference"] = df[ind].map(mapping)
    d["n/s"] = d["reference"].map(ns_map)
    d["individual"] = ind[:-2]
    d["haplotype"] = ind[-1:]
    return pd.DataFrame(d)

def create_paint_df_ind(df, ind, mapping, ns_map):
    d = {}
    d["pos"] = df.spos
    d["end_pos"] = df.epos -1
    d["length"] = df.epos - df.spos
    d["reference"] = df[ind].map(mapping)
    d["n/s"] = d["reference"].map(ns_map)
    d["individual"] = ind[:-2]
    d["haplotype"] = ind[-1:]
    return pd.DataFrame(d)

@window(size=100000)
def north_sum(df):
    return (df.end-df.start).sum()

def add_dummy(c_df):
    inds = c_df.individual.unique()
    dummy_df = pd.DataFrame({"individual": np.repeat(inds, 2)})
    dummy_df["haplotype"] = pd.Series(["0", "1"]*len(inds))
    dummy_df["end"], dummy_df["start"] = c_df.end.max(), c_df.end.max()
    dummy_df["chrom"] = c_df.chrom.unique()[0]
    return dummy_df

First running it for the "distant" setup with Kindae/Ursinus and Hamadryas/Papio as references, then for the "close" setup with Mikumi and Serengeti as the references.

In [3]:
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering.txt", sep =" ")
rfmix_path = "../steps/rfmix_gen100/tanzania_focus/"
mapping_df = pd.read_csv(rfmix_path + "ref_names.txt", sep = "\t", names=["ID", "Origin"])

In [4]:
meta_data_samples_Sci = meta_data_samples.copy()
for i, row in meta_data_samples_Sci.iterrows():
    if row.PGDP_ID[0] != "P":
        meta_data_samples_Sci.at[i, "PGDP_ID"] = "Sci_"+str(row.PGDP_ID)
meta_data_samples_Sci.to_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ", index=False)

In [5]:
meta_data_samples_Sci

,PGDP_ID,Provider_ID,Provider,Genus,Species,Origin,Sex,address,longitude,latitude,callset_index,C_origin,x_missing
0,Sci_16066,16066_3130,Rogers,Papio,cynocephalus,"Mikumi, Tanzania",F,"Mikumi, Kilosa, Morogoro, Coastal Zone, Tanzania",37,-7.4,0,"Cynocephalus, Central Tanzania",0.0139
1,Sci_16098,16098_5026,Rogers,Papio,cynocephalus,"Mikumi, Tanzania",F,"Mikumi, Kilosa, Morogoro, Coastal Zone, Tanzania",37,-7.4,1,"Cynocephalus, Central Tanzania",0.0138
2,Sci_30877,30877_3426,James Else,Papio,anubis,"Aberdare, Kenya",M,"Aberdare National Park, Nyeri, Central Kenya, ...",36.7,-0.41,2,"Anubis, Kenya",0.0156
3,Sci_30977,30977_3373,James Else,Papio,anubis,"Aberdare, Kenya",F,"Aberdare National Park, Nyeri, Central Kenya, ...",36.7,-0.41,3,"Anubis, Kenya",0.01
4,Sci_34449,34449_BZ11022,Rogers/Jolly/Phillips-Conroy,Papio,kindae,"Chunga, Zambia",F,"Chunga, Mumbwa District, Central Province, Zambia",26,-15.1,4,"Kindae, Zambia",0.0135
...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,PD_0790,34418_BZ11065,Rogers/Jolly/Phillips-Conroy,Papio,kindae,"Chunga, Zambia",F,"Chunga, Mumbwa District, Central Province, Zambia",26,-15.1,222,"Kindae, Zambia",0.00983
223,PD_0791,34419_BZ11066,Rogers/Jolly/Phillips-Conroy,Papio,kindae,"Chunga, Zambia",F,"Chunga, Mumbwa District, Central Province, Zambia",26,-15.1,223,"Kindae, Zambia",0.00967
224,PD_0792,34420_BZ11067,Rogers/Jolly/Phillips-Conroy,Papio,kindae,"Chunga, Zambia",F,"Chunga, Mumbwa District, Central Province, Zambia",26,-15.1,224,"Kindae, Zambia",0.0101
225,PD_0793,34422_BZ11070,Rogers/Jolly/Phillips-Conroy,Papio,kindae,"Chunga, Zambia",M,"Chunga, Mumbwa District, Central Province, Zambia",26,-15.1,225,"Kindae, Zambia",0.0143


In [6]:
pop_mapping = {}
o_order = sorted((mapping_df.Origin.unique()))
for o in range(len((mapping_df.Origin.unique()))):
    pop_mapping[o_order[o]] = o
    
north_south_mapping = {'Hamadryas, Ethiopia': 1,
 'Kindae, Zambia': -1,
 'Papio, Senegal': 1,
 'Ursinus, Zambia': -1}

In [7]:
df_l = []
for chrom in ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "female_chrX", "female_ref_chrX"]:
    print(chrom)
    input_file = rfmix_path + "{}.msp.tsv".format(chrom)
    chr_df = pd.read_csv(input_file, sep = "\t", header=1)
    file = open(input_file, 'r')
    line1 = file.readline()
    file.close()
    number_subpop_mapping = {}
    for subpop in line1.strip().split("\t"):
        # Splitting to create lists containing two values - subpop and number.
        subpop_number = subpop.split(": ")[-1].split("=")
        number_subpop_mapping[int(subpop_number[1])] = subpop_number[0]
    for hap in chr_df.columns[6:]:
        paint_df = create_paint_df_ind_compressed(chr_df, hap, number_subpop_mapping, north_south_mapping)
        paint_df["chrom"] = "{}".format(chrom)
        df_l.append(paint_df)
length_df = pd.concat(df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
all_chrX
female_chrX
female_ref_chrX


In [8]:
for c in length_df.chrom.unique():
    output_file = rfmix_path + "{}.windows.txt".format(c)
    print(output_file)
    if os.path.exists(output_file) == False:
        c_df = length_df.loc[(length_df.chrom == c)]
        c_df = c_df.rename(columns={"pos": "start", "end_pos": "end"})
        dummy_added = pd.concat([c_df, add_dummy(c_df)]).reset_index()
        dummy_added = dummy_added.loc[(dummy_added["n/s"] != -1)].copy()
        df = dummy_added.groupby(['chrom', 'individual', 'haplotype'])[["end", "start"]].apply(north_sum).reset_index(drop=True, level=-1).reset_index()
        df.to_csv(output_file, index=False, sep="\t")

../steps/rfmix_gen100/tanzania_focus/chr1.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr2.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr3.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr4.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr5.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr6.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr7.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr8.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr9.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr10.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr11.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr12.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr13.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr14.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr15.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr16.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr17.windows.txt
../steps/rfmix_gen100/tanzania_focus/chr18.windows.txt
../steps/rfmix_gen1

And for the close pops

In [9]:
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ")
rfmix_path = "../steps/rfmix_gen100/tanzania_close_focus/"
mapping_df = pd.read_csv(rfmix_path + "ref_names.txt", sep = "\t", names=["ID", "Origin"])

pop_mapping = {}
o_order = sorted((mapping_df.Origin.unique()))
for o in range(len((mapping_df.Origin.unique()))):
    pop_mapping[o_order[o]] = o
north_south_mapping = {'Serengeti, Tanzania': 1,
 'Mikumi, Tanzania': -1}

In [10]:
df_l = []
for chrom in ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "female_chrX", "female_ref_chrX"]:
    print(chrom)
    input_file = rfmix_path + "{}.msp.tsv".format(chrom)
    chr_df = pd.read_csv(input_file, sep = "\t", header=1)
    file = open(input_file, 'r')
    line1 = file.readline()
    file.close()
    number_subpop_mapping = {}
    for subpop in line1.strip().split("\t"):
        # Splitting to create lists containing two values - subpop and number.
        subpop_number = subpop.split(": ")[-1].split("=")
        number_subpop_mapping[int(subpop_number[1])] = subpop_number[0]
    for hap in chr_df.columns[6:]:
        paint_df = create_paint_df_ind_compressed(chr_df, hap, number_subpop_mapping, north_south_mapping)
        paint_df["chrom"] = "{}".format(chrom)
        df_l.append(paint_df)
length_df = pd.concat(df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
all_chrX
female_chrX
female_ref_chrX


In [11]:
for c in length_df.chrom.unique():
    output_file = rfmix_path + "{}.windows.txt".format(c)
    print(output_file)
    if os.path.exists(output_file) == False:
        c_df = length_df.loc[(length_df.chrom == c)]
        c_df = c_df.rename(columns={"pos": "start", "end_pos": "end"})
        dummy_added = pd.concat([c_df, add_dummy(c_df)]).reset_index()
        dummy_added = dummy_added.loc[(dummy_added["n/s"] != -1)].copy()
        df = dummy_added.groupby(['chrom', 'individual', 'haplotype'])[["end", "start"]].apply(north_sum).reset_index(drop=True, level=-1).reset_index()
        df.to_csv(output_file, index=False, sep="\t")

../steps/rfmix_gen100/tanzania_close_focus/chr1.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr2.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr3.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr4.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr5.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr6.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr7.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr8.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr9.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr10.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr11.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr12.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr13.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr14.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr15.windows.txt
../steps/rfmix_gen100/tanzania_close_focus/chr16.windows.txt
../steps/rfmix_gen100/tanzania_cl

And lastly for the ethiopian olives.

In [12]:
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ")
rfmix_path = "../steps/rfmix_gen100/eth_olive_focus/"
mapping_df = pd.read_csv(rfmix_path + "ref_names.txt", sep = "\t", names=["ID", "Origin"])

pop_mapping = {}
o_order = sorted((mapping_df.Origin.unique()))
for o in range(len((mapping_df.Origin.unique()))):
    pop_mapping[o_order[o]] = o
north_south_mapping = {'Hamadryas, Ethiopia': 1,
 'Cynocephalus, Central Tanzania': 0,
 'Papio, Senegal': 0,
 'Anubis, Tanzania': 0}

In [13]:
df_l = []
for chrom in ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "female_chrX", "female_ref_chrX"]:
    print(chrom)
    input_file = rfmix_path + "{}.msp.tsv".format(chrom)
    chr_df = pd.read_csv(input_file, sep = "\t", header=1)
    file = open(input_file, 'r')
    line1 = file.readline()
    file.close()
    number_subpop_mapping = {}
    for subpop in line1.strip().split("\t"):
        # Splitting to create lists containing two values - subpop and number.
        subpop_number = subpop.split(": ")[-1].split("=")
        number_subpop_mapping[int(subpop_number[1])] = subpop_number[0]
    for hap in chr_df.columns[6:]:
        paint_df = create_paint_df_ind_compressed(chr_df, hap, number_subpop_mapping, north_south_mapping)
        paint_df["chrom"] = "{}".format(chrom)
        df_l.append(paint_df)
length_df = pd.concat(df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
all_chrX
female_chrX
female_ref_chrX


In [14]:
for c in length_df.chrom.unique():
    output_file = rfmix_path + "{}.windows.txt".format(c)
    print(output_file)
    if os.path.exists(output_file) == False:
        c_df = length_df.loc[(length_df.chrom == c)]
        c_df = c_df.rename(columns={"pos": "start", "end_pos": "end"})
        dummy_added = pd.concat([c_df, add_dummy(c_df)]).reset_index()
        dummy_added = dummy_added.loc[(dummy_added["n/s"] != -1)].copy()
        df = dummy_added.groupby(['chrom', 'individual', 'haplotype'])[["end", "start"]].apply(north_sum).reset_index(drop=True, level=-1).reset_index()
        df.to_csv(output_file, index=False, sep="\t")

../steps/rfmix_gen100/eth_olive_focus/chr1.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr2.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr3.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr4.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr5.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr6.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr7.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr8.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr9.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr10.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr11.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr12.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr13.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr14.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr15.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr16.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr17.windows.txt
../steps/rfmix_gen100/eth_olive_focus/chr18.windows.txt
.

I'm unsure if the reanalysis reference setting hinders the analysis, so I will try without.

In [15]:
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ")
rfmix_path = "../steps/rfmix_gen100/eth_olive_focus_plain/"
mapping_df = pd.read_csv("../steps/rfmix_gen100/eth_olive_focus/" + "ref_names.txt", sep = "\t", names=["ID", "Origin"])

pop_mapping = {}
o_order = sorted((mapping_df.Origin.unique()))
for o in range(len((mapping_df.Origin.unique()))):
    pop_mapping[o_order[o]] = o
north_south_mapping = {'Hamadryas, Ethiopia': 1,
 'Cynocephalus, Central Tanzania': 0,
 'Papio, Senegal': 0,
 'Anubis, Tanzania': 0}

In [16]:
df_l = []
for chrom in ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "female_chrX", "female_ref_chrX"]:
    print(chrom)
    input_file = rfmix_path + "{}.msp.tsv".format(chrom)
    chr_df = pd.read_csv(input_file, sep = "\t", header=1)
    file = open(input_file, 'r')
    line1 = file.readline()
    file.close()
    number_subpop_mapping = {}
    for subpop in line1.strip().split("\t"):
        # Splitting to create lists containing two values - subpop and number.
        subpop_number = subpop.split(": ")[-1].split("=")
        number_subpop_mapping[int(subpop_number[1])] = subpop_number[0]
    for hap in chr_df.columns[6:]:
        paint_df = create_paint_df_ind_compressed(chr_df, hap, number_subpop_mapping, north_south_mapping)
        paint_df["chrom"] = "{}".format(chrom)
        df_l.append(paint_df)
length_df = pd.concat(df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
all_chrX
female_chrX
female_ref_chrX


In [17]:
for c in length_df.chrom.unique():
    output_file = rfmix_path + "{}.windows.txt".format(c)
    print(output_file)
    if os.path.exists(output_file) == False:
        c_df = length_df.loc[(length_df.chrom == c)]
        c_df = c_df.rename(columns={"pos": "start", "end_pos": "end"})
        dummy_added = pd.concat([c_df, add_dummy(c_df)]).reset_index()
        dummy_added = dummy_added.loc[(dummy_added["n/s"] != -1)].copy()
        df = dummy_added.groupby(['chrom', 'individual', 'haplotype'])[["end", "start"]].apply(north_sum).reset_index(drop=True, level=-1).reset_index()
        df.to_csv(output_file, index=False, sep="\t")

../steps/rfmix_gen100/eth_olive_focus_plain/chr1.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr2.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr3.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr4.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr5.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr6.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr7.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr8.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr9.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr10.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr11.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr12.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr13.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr14.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr15.windows.txt
../steps/rfmix_gen100/eth_olive_focus_plain/chr16.windows.txt
../steps/rfmix_ge

In [18]:
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ")
rfmix_path = "../steps/rfmix_gen100/tanzania_focus_plain/"
mapping_df = pd.read_csv("../steps/rfmix_gen100/tanzania_focus/" + "ref_names.txt", sep = "\t", names=["ID", "Origin"])
pop_mapping = {}
o_order = sorted((mapping_df.Origin.unique()))
for o in range(len((mapping_df.Origin.unique()))):
    pop_mapping[o_order[o]] = o
    
north_south_mapping = {'Hamadryas, Ethiopia': 1,
 'Kindae, Zambia': -1,
 'Papio, Senegal': 1,
 'Ursinus, Zambia': -1}

In [19]:
df_l = []
for chrom in ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "female_chrX", "female_ref_chrX"]:
    print(chrom)
    input_file = rfmix_path + "{}.msp.tsv".format(chrom)
    chr_df = pd.read_csv(input_file, sep = "\t", header=1)
    file = open(input_file, 'r')
    line1 = file.readline()
    file.close()
    number_subpop_mapping = {}
    for subpop in line1.strip().split("\t"):
        # Splitting to create lists containing two values - subpop and number.
        subpop_number = subpop.split(": ")[-1].split("=")
        number_subpop_mapping[int(subpop_number[1])] = subpop_number[0]
    for hap in chr_df.columns[6:]:
        paint_df = create_paint_df_ind_compressed(chr_df, hap, number_subpop_mapping, north_south_mapping)
        paint_df["chrom"] = "{}".format(chrom)
        df_l.append(paint_df)
length_df = pd.concat(df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
all_chrX
female_chrX
female_ref_chrX


In [20]:
for c in length_df.chrom.unique():
    output_file = rfmix_path + "{}.windows.txt".format(c)
    print(output_file)
    if os.path.exists(output_file) == False:
        c_df = length_df.loc[(length_df.chrom == c)]
        c_df = c_df.rename(columns={"pos": "start", "end_pos": "end"})
        dummy_added = pd.concat([c_df, add_dummy(c_df)]).reset_index()
        dummy_added = dummy_added.loc[(dummy_added["n/s"] != -1)].copy()
        df = dummy_added.groupby(['chrom', 'individual', 'haplotype'])[["end", "start"]].apply(north_sum).reset_index(drop=True, level=-1).reset_index()
        df.to_csv(output_file, index=False, sep="\t")

../steps/rfmix_gen100/tanzania_focus_plain/chr1.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr2.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr3.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr4.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr5.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr6.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr7.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr8.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr9.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr10.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr11.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr12.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr13.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr14.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr15.windows.txt
../steps/rfmix_gen100/tanzania_focus_plain/chr16.windows.txt
../steps/rfmix_gen100/tanzania_fo

In [48]:
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ")
rfmix_path = "../steps/rfmix_gen100/tanzania_close_focus_plain/"
mapping_df = pd.read_csv("../steps/rfmix_gen100/tanzania_close_focus/" + "ref_names.txt", sep = "\t", names=["ID", "Origin"])

pop_mapping = {}
o_order = sorted((mapping_df.Origin.unique()))
for o in range(len((mapping_df.Origin.unique()))):
    pop_mapping[o_order[o]] = o
north_south_mapping = {'Serengeti, Tanzania': 1,
 'Mikumi, Tanzania': -1}

In [49]:
df_l = []
for chrom in ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "female_chrX", "female_ref_chrX"]:
    print(chrom)
    input_file = rfmix_path + "{}.msp.tsv".format(chrom)
    chr_df = pd.read_csv(input_file, sep = "\t", header=1)
    file = open(input_file, 'r')
    line1 = file.readline()
    file.close()
    number_subpop_mapping = {}
    for subpop in line1.strip().split("\t"):
        # Splitting to create lists containing two values - subpop and number.
        subpop_number = subpop.split(": ")[-1].split("=")
        number_subpop_mapping[int(subpop_number[1])] = subpop_number[0]
    for hap in chr_df.columns[6:]:
        paint_df = create_paint_df_ind_compressed(chr_df, hap, number_subpop_mapping, north_south_mapping)
        paint_df["chrom"] = "{}".format(chrom)
        df_l.append(paint_df)
length_df = pd.concat(df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
all_chrX
female_chrX
female_ref_chrX


In [50]:
pd0213_df = length_df.loc[length_df.individual == "PD_0213"].copy()
pd0213_df.groupby(["reference"])[["length"]].sum()/pd0213_df.length.sum()

,length
reference,
"Mikumi, Tanzania",0.912
"Serengeti, Tanzania",0.0882


In [52]:
pd0213_df.loc[pd0213_df.chrom == "all_chrX"].groupby(["reference"])[["length"]].sum()/(pd0213_df.loc[pd0213_df.chrom == "all_chrX"].length.sum())

,length
reference,
"Mikumi, Tanzania",0.901
"Serengeti, Tanzania",0.099


In [23]:
for c in length_df.chrom.unique():
    output_file = rfmix_path + "{}.windows.txt".format(c)
    print(output_file)
    if os.path.exists(output_file) == False:
        c_df = length_df.loc[(length_df.chrom == c)]
        c_df = c_df.rename(columns={"pos": "start", "end_pos": "end"})
        dummy_added = pd.concat([c_df, add_dummy(c_df)]).reset_index()
        dummy_added = dummy_added.loc[(dummy_added["n/s"] != -1)].copy()
        df = dummy_added.groupby(['chrom', 'individual', 'haplotype'])[["end", "start"]].apply(north_sum).reset_index(drop=True, level=-1).reset_index()
        df.to_csv(output_file, index=False, sep="\t")

../steps/rfmix_gen100/tanzania_close_focus_plain/chr1.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr2.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr3.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr4.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr5.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr6.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr7.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr8.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr9.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr10.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr11.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr12.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr13.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr14.windows.txt
../steps/rfmix_gen100/tanzania_close_focus_plain/chr15.windows.txt
../s

In [58]:
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ")
rfmix_path = "../steps/rfmix_gen100/tanzania_pure_focus/"
mapping_df = pd.read_csv("../steps/rfmix_gen100/tanzania_close_focus/" + "ref_names.txt", sep = "\t", names=["ID", "Origin"])

pop_mapping = {}
o_order = sorted((mapping_df.Origin.unique()))
for o in range(len((mapping_df.Origin.unique()))):
    pop_mapping[o_order[o]] = o
north_south_mapping = {'Gog Woreda, Gambella region, Ethiopia': 1,
                         'Mikumi, Tanzania': -1,
                          'Mahale, Tanzania': -1}

In [59]:
df_l = []
for chrom in ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX", "female_chrX", "female_ref_chrX"]:
    print(chrom)
    input_file = rfmix_path + "{}.msp.tsv".format(chrom)
    chr_df = pd.read_csv(input_file, sep = "\t", header=1)
    file = open(input_file, 'r')
    line1 = file.readline()
    file.close()
    number_subpop_mapping = {}
    for subpop in line1.strip().split("\t"):
        # Splitting to create lists containing two values - subpop and number.
        subpop_number = subpop.split(": ")[-1].split("=")
        number_subpop_mapping[int(subpop_number[1])] = subpop_number[0]
    for hap in chr_df.columns[6:]:
        paint_df = create_paint_df_ind_compressed(chr_df, hap, number_subpop_mapping, north_south_mapping)
        paint_df["chrom"] = "{}".format(chrom)
        df_l.append(paint_df)
length_df = pd.concat(df_l)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
all_chrX
female_chrX
female_ref_chrX


In [60]:
pd0213_df = length_df.loc[length_df.individual == "PD_0213"].copy()

In [61]:
pd0213_df.groupby(["reference"])[["length"]].sum()/pd0213_df.length.sum()

,length
reference,
"Gog Woreda, Gambella region, Ethiopia",0.0742
"Mahale, Tanzania",0.129
"Mikumi, Tanzania",0.797


In [62]:
pd0213_df.loc[pd0213_df.chrom == "all_chrX"].groupby(["reference"])[["length"]].sum()/(pd0213_df.loc[pd0213_df.chrom == "all_chrX"].length.sum())

,length
reference,
"Gog Woreda, Gambella region, Ethiopia",0.064
"Mahale, Tanzania",0.187
"Mikumi, Tanzania",0.749


In [65]:
meta_data_samples.loc[meta_data_samples.Origin == "Udzungwa, Tanzania"].PGDP_ID

34    PD_0223
35    PD_0224
36    PD_0225
37    PD_0226
38    PD_0227
Name: PGDP_ID, dtype: object

In [66]:
udzungwa_df = length_df.loc[length_df.individual.isin(meta_data_samples.loc[meta_data_samples.Origin == "Udzungwa, Tanzania"].PGDP_ID)].copy()

In [67]:
udzungwa_df.loc[udzungwa_df.chrom == "all_chrX"].groupby(["reference"])[["length"]].sum()/(udzungwa_df.loc[udzungwa_df.chrom == "all_chrX"].length.sum())

,length
reference,
"Gog Woreda, Gambella region, Ethiopia",0.0115
"Mahale, Tanzania",0.03
"Mikumi, Tanzania",0.959


In [68]:
udzungwa_df.groupby(["reference"])[["length"]].sum()/(udzungwa_df.length.sum())

,length
reference,
"Gog Woreda, Gambella region, Ethiopia",0.0108
"Mahale, Tanzania",0.0184
"Mikumi, Tanzania",0.971
